# Analysis & Analytics

## Data Acquisition

### Prepartion

#### Imports

In [1]:
from nyc_taxi_routes.notebook import *
setup_plotting()

✓  wgnd theme activated (matplotlib · seaborn)

#### Constants

In [2]:
DATA_PATH = PATHS["processed"]
DATA_FILE= 'ny-taxi-routes_prep.parquet'

### Data Gathering

In [3]:
df_prep= df = pd.read_parquet(DATA_PATH / DATA_FILE, engine='pyarrow')

df_final= df_prep.copy()

df_final.head()

,pickup_weekday,pickup_hour,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count,trip_distance,fare_amount,tip_amount,tolls_amount,payment_type,departure,arrival,route,total_yield,price_per_mile,has_tolls,time_slot,is_weekend,trip_distance_log,fare_amount_log,total_yield_log,price_per_mile_log
0,3,19,-73.789970,40.646660,-74.005051,40.748081,1,18.610001,52.0,10.00,5.54,1,JFK,NYC,JFK-NYC,62.00,3.331542,1,Evening Rush,0,2.976040,3.970292,4.143135,1.465924
1,5,3,-73.986237,40.746513,-73.996796,40.742504,1,0.990000,5.0,1.00,0.00,1,NYC,NYC,NYC-NYC,6.00,6.060606,0,Night,1,0.688135,1.791759,1.945910,1.954531
2,4,20,-73.874634,40.773960,-73.959923,40.762802,3,9.250000,26.5,8.34,5.54,1,NYC,NYC,NYC-NYC,34.84,3.766486,1,Evening Rush,0,2.327278,3.314186,3.579065,1.561609
3,5,2,-73.952477,40.772064,-73.949371,40.675156,1,9.200000,28.0,0.00,0.00,2,NYC,NYC,NYC-NYC,28.00,3.043478,0,Night,1,2.322388,3.367296,3.367296,1.397105
4,4,21,-73.988281,40.764488,-73.996513,40.753239,1,0.900000,5.0,1.26,0.00,1,NYC,NYC,NYC-NYC,6.26,6.955556,0,Late Night,0,0.641854,1.791759,1.982380,2.073871


## Business Questions

**1)** [Anteil JFK Departures](###-Anteil-JFK-Departures)   
Wie hoch ist der Anteil an Taxis, die vom Flughafen (JFK) aus gebucht werden insgesamt? 

**2)** Wo werden Taxis in New York genommen? Erstelle eine Visualisierung der Startpunkte der Taxifahrten.  

**3)** Wie hoch ist der Anteil an Taxis, die vom Flughafen aus gebucht werden pro Wochentag? An welchem Wochentag gibt es den höchsten Anteil und wann den niedrigsten?

**4)** Erstelle eine Visualisierung, anhand derer man sieht, welchen Anteil ein **Wochentag** an der Anzahl der Fahrten insgesamt hat. Dies soll sowohl für die Fahrten vom Flughafen aus gemacht werden als auch für das Gesamtset.  

**5)** Erstelle eine Visualisierung, anhand derer man sieht, welchen Anteil eine **Uhrzeit** an der Anzahl der Fahrten insgesamt hat. Dies soll sowohl für die Fahrten vom Flughafen aus gemacht werden als auch für das Gesamtset.  

### Anteil JFK Departures

In [4]:
# NEW DATA FRAME FOR ROUTES WITH GROUP BY

inspect(df_final,['dimensions'])

section_header('Routes Overview')
df_routes = df_final.groupby(["route"], observed=False ).agg(
    trip_sum=("trip_distance", "count"),
    dist_sum=("trip_distance", "sum") ,
    fare_sum=("fare_amount", "sum")
)
df_routes['trip_pct'] = (df_routes['trip_sum'] / df_routes['trip_sum'].sum()) * 100
df_routes['fare_pct'] = (df_routes['fare_sum'] / df_routes['fare_sum'].sum()) * 100
df_routes['dist_pct'] = (df_routes['dist_sum'] / df_routes['dist_sum'].sum()) * 100
show_df(df_routes)

section_header('JFK Depatures')
df_jfk_dep = df_final.groupby("departure", observed=False).size().reset_index(name='cnt')
df_jfk_dep['pct'] = (df_jfk_dep['cnt'] / df_jfk_dep['cnt'].sum()) * 100
show_df(df_jfk_dep)

total_trips = df_final.shape[0]
df_dep_arr_jfk_sum = df_routes.loc[['JFK-JFK', 'JFK-NYC', 'JFK-OTHER', 'NYC-JFK'], :]["trip_sum"].sum()
dep_arr_jfk_pct = round( (df_dep_arr_jfk_sum * 100) /  total_trips,2)


success(f'Anteil der Abfahrten am JFK beträgt {df_jfk_dep.loc[0, "pct"]/100:.2%}')
log(f'\n- Gesamtanzahl aller Fahrten: {total_trips}')
log(f"- Gesamtnzahl der Fahrten zum und vom Flughafen: {df_dep_arr_jfk_sum}")
log(f"- Gesamtanteil der Fahrten zum und vom Flughafen:  {dep_arr_jfk_pct}%")




───  DIMENSIONS  ─────────────────────────────────────────────


,metric,count,pct
0,rows,293369,
1,columns,24,
2,duplicates,0,0.0%
3,empty rows (all NaN),0,0.0%
4,empty cols (all NaN),0,0.0%



───  ROUTES OVERVIEW  ────────────────────────────────────────


,trip_sum,dist_sum,fare_sum,trip_pct,fare_pct,dist_pct
route,,,,,,
JFK-JFK,143,577.06,2951.06,0.05%,0.08%,0.07%
JFK-NYC,5445,89246.93,253069.50,1.86%,7.01%,10.65%
JFK-OTHER,61,1198.00,3928.00,0.02%,0.11%,0.14%
NYC-JFK,1964,34670.84,100673.50,0.67%,2.79%,4.14%
NYC-NYC,285174,702059.25,3207394.25,97.21%,88.90%,83.77%
NYC-OTHER,551,10092.55,38407.95,0.19%,1.06%,1.20%
OTHER-NYC,1,25.20,73.00,0.00%,0.00%,0.00%
OTHER-OTHER,30,170.47,1194.75,0.01%,0.03%,0.02%



───  JFK DEPATURES  ──────────────────────────────────────────


,departure,cnt,pct
0,JFK,5649,1.93%
1,NYC,287689,98.06%
2,OTHER,31,0.01%


✓  Anteil der Abfahrten am JFK beträgt 1.93%

- Gesamtanzahl aller Fahrten: 293369
- Gesamtnzahl der Fahrten zum und vom Flughafen: 7613
- Gesamtanteil der Fahrten zum und vom Flughafen:  2.6%


### Visualisierung der Pickup-Standorte (Frage 4)


In [5]:
show_geo_locations_map(df_final)
plt.savefig(PATHS['figures'] / 'pickup_locations_map.png', bbox_inches='tight', dpi=150)
plt.show()

/Users/kaywiegand/Workspace/nyc-taxi-routes/src/nyc_taxi_routes/visualization/geo_locations.py:74: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


/var/folders/jh/b553h44j08x_jr8xwh9jbc5r0000gn/T/ipykernel_17116/2451584564.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


![Pickup Locations](../public/img/pickup_locations_map.png)

Die meisten Fahrten starten in Manhattan (dichter Cluster), JFK-Abfahrten (rot) bilden einen
klar abgegrenzten zweiten Cluster am Flughafen selbst.


### JFK-Anteil pro Wochentag (Frage 5)


In [6]:
WEEKDAY_LABELS = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']

section_header('JFK Departure Share by Weekday')
df_jfk_wd = df_final.groupby('pickup_weekday', observed=True).agg(
    total=('departure', 'size'),
    jfk=('departure', lambda s: (s == 'JFK').sum())
)
df_jfk_wd['jfk_share_pct'] = (df_jfk_wd['jfk'] / df_jfk_wd['total'] * 100).round(2)
df_jfk_wd.index = [WEEKDAY_LABELS[i] for i in df_jfk_wd.index]
show_df(df_jfk_wd)

highest = df_jfk_wd['jfk_share_pct'].idxmax()
lowest = df_jfk_wd['jfk_share_pct'].idxmin()
success(f"Höchster JFK-Anteil: {highest} ({df_jfk_wd['jfk_share_pct'][highest]}%) — niedrigster: {lowest} ({df_jfk_wd['jfk_share_pct'][lowest]}%)")

plt.figure(figsize=(8, 5))
plt.bar(df_jfk_wd.index, df_jfk_wd['jfk_share_pct'], color=cfg.ACTIVE_PALETTE[0])
plt.title('JFK Departure Share by Weekday')
plt.ylabel('% of trips departing from JFK')
plt.grid(True, axis='y', alpha=0.3)
plt.savefig(PATHS['figures'] / 'jfk_share_by_weekday.png', bbox_inches='tight', dpi=150)
plt.show()


───  JFK DEPARTURE SHARE BY WEEKDAY  ─────────────────────────


,total,jfk,jfk_share_pct
Mon,44723,1164,2.60%
Tue,38677,774,2.00%
Wed,39972,682,1.71%
Thu,42397,769,1.81%
Fri,44019,820,1.86%
Sat,44669,643,1.44%
Sun,38912,797,2.05%


2026-07-11 12:29:36  INFO      matplotlib.category  Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.


2026-07-11 12:29:36  INFO      matplotlib.category  Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.


✓  Höchster JFK-Anteil: Mon (2.6%) — niedrigster: Sat (1.44%)


/var/folders/jh/b553h44j08x_jr8xwh9jbc5r0000gn/T/ipykernel_17116/1271875694.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Wochentags-Verteilung: Gesamt vs. JFK (Frage 6)


In [7]:
section_header('Weekday Distribution — Overall vs. JFK')
total_by_wd = df_final['pickup_weekday'].value_counts(normalize=True).sort_index() * 100
jfk_by_wd = df_final[df_final['departure'] == 'JFK']['pickup_weekday'].value_counts(normalize=True).sort_index() * 100
compare_wd = pd.DataFrame({'overall_pct': total_by_wd, 'jfk_pct': jfk_by_wd}).fillna(0).round(2)
compare_wd.index = [WEEKDAY_LABELS[i] for i in compare_wd.index]
show_df(compare_wd)

x = range(len(compare_wd))
width = 0.35
plt.figure(figsize=(9, 5))
plt.bar([i - width/2 for i in x], compare_wd['overall_pct'], width, label='All trips', color=cfg.ACTIVE_PALETTE[4])
plt.bar([i + width/2 for i in x], compare_wd['jfk_pct'], width, label='JFK departures', color=cfg.ACTIVE_PALETTE[0])
plt.xticks(list(x), compare_wd.index)
plt.ylabel('% of trips (within group)')
plt.title('Weekday Distribution — All Trips vs. JFK Departures')
plt.legend()
plt.grid(True, axis='y', alpha=0.3)
plt.savefig(PATHS['figures'] / 'weekday_distribution_overall_vs_jfk.png', bbox_inches='tight', dpi=150)
plt.show()


───  WEEKDAY DISTRIBUTION — OVERALL VS. JFK  ─────────────────


,overall_pct,jfk_pct
Mon,15.24%,20.61%
Tue,13.18%,13.70%
Wed,13.63%,12.07%
Thu,14.45%,13.61%
Fri,15.00%,14.52%
Sat,15.23%,11.38%
Sun,13.26%,14.11%


/var/folders/jh/b553h44j08x_jr8xwh9jbc5r0000gn/T/ipykernel_17116/3599343830.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Uhrzeit-Verteilung: Gesamt vs. JFK (Frage 7)


In [8]:
section_header('Hourly Distribution — Overall vs. JFK')
total_by_hr = df_final['pickup_hour'].value_counts(normalize=True).sort_index() * 100
jfk_by_hr = df_final[df_final['departure'] == 'JFK']['pickup_hour'].value_counts(normalize=True).sort_index() * 100
compare_hr = pd.DataFrame({'overall_pct': total_by_hr, 'jfk_pct': jfk_by_hr}).fillna(0).round(2)
show_df(compare_hr)

success(f"Peak-Stunde gesamt: {compare_hr['overall_pct'].idxmax()}h — Peak-Stunde JFK: {compare_hr['jfk_pct'].idxmax()}h")

plt.figure(figsize=(10, 5))
plt.plot(compare_hr.index, compare_hr['overall_pct'], label='All trips', color=cfg.ACTIVE_PALETTE[4], marker='o', markersize=3)
plt.plot(compare_hr.index, compare_hr['jfk_pct'], label='JFK departures', color=cfg.ACTIVE_PALETTE[0], marker='o', markersize=3)
plt.xticks(range(0, 24))
plt.xlabel('Hour of day')
plt.ylabel('% of trips (within group)')
plt.title('Hourly Distribution — All Trips vs. JFK Departures')
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig(PATHS['figures'] / 'hourly_distribution_overall_vs_jfk.png', bbox_inches='tight', dpi=150)
plt.show()


───  HOURLY DISTRIBUTION — OVERALL VS. JFK  ──────────────────


,overall_pct,jfk_pct
pickup_hour,,
0,3.46%,3.93%
1,2.44%,1.96%
2,1.89%,0.53%
3,1.32%,0.41%
4,1.05%,0.48%
5,0.97%,2.14%
6,2.22%,4.64%
7,3.75%,4.21%
8,4.61%,2.85%


✓  Peak-Stunde gesamt: 18h — Peak-Stunde JFK: 17h


/var/folders/jh/b553h44j08x_jr8xwh9jbc5r0000gn/T/ipykernel_17116/4093508998.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Next Step

→ Zusammenfassung, Interpretation und Flottenempfehlung: [`04_insights.ipynb`](04_insights.ipynb)
